# Step 4: RQ2 Logistic Redesigned

This redesigned RQ2 notebook tests whether LLMs can infer model-relevant features from borrower input values alone. The prompt uses readable feature names only, while the parser maps readable/raw aliases to the canonical feature names used for evaluation. A feature-name-only control is included to estimate how much performance comes from generic credit-risk priors rather than sample-specific values.


## 1. Setup

Load shared packages and define constants.


In [ ]:
from pathlib import Path
import json
import os
import pickle
import random
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
from sklearn.metrics import precision_recall_curve

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 80)

RANDOM_STATE = 42
N_SAMPLES = 500
BATCH_SIZE = 20
TOP_N_FEATURES = 24
DEFAULT_THRESHOLD = 0.556
RECREATE_SAMPLES = True


## 2. Paths

Use the Google Drive project folder when mounted in Colab; otherwise use the current local project folder.


In [ ]:
PROJECT_ROOT = Path.cwd()
COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_ClubLending")


def find_local_project_root(start):
    """Find the LLM_Exapliner project root when running from Code/ or the project root."""
    start = Path(start).resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "Data" / "ModifiedData").exists() and (candidate / "Result").exists():
            return candidate
    return start.parent if start.name == "Code" else start


BASE_DIR = COLAB_PROJECT_ROOT if COLAB_PROJECT_ROOT.exists() else find_local_project_root(PROJECT_ROOT)

MODIFIED_DATA_DIR = BASE_DIR / "Data" / "ModifiedData"
RESULT_ROOT = BASE_DIR / "Result"
RESULT_PROJECT_DIR = RESULT_ROOT / "LendingClub" if (RESULT_ROOT / "LendingClub").exists() else RESULT_ROOT
BASELINE_MODEL_DIR = RESULT_PROJECT_DIR / "BaselineModel"
SHAP_DIR = RESULT_PROJECT_DIR / "SHAP"
LLM_DIR = RESULT_PROJECT_DIR / "LLM"
RQ2_LOGISTIC_DIR = LLM_DIR / "RQ2_Logistic_Redesigned"
RQ2_LOGISTIC_DIR.mkdir(parents=True, exist_ok=True)


def first_existing(directory, filenames):
    for filename in filenames:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    return directory / filenames[0]


SPLIT_DATA_PATH = MODIFIED_DATA_DIR / "LC_split.pkl"
LOGISTIC_MODEL_PATH = first_existing(BASELINE_MODEL_DIR, ["logistic_pipeline.pkl"])
LOGISTIC_CONTRIB_PATH = first_existing(SHAP_DIR, ["Logistic_Contrib.xlsx"])
SAMPLE_PATH = LLM_DIR / "500_samples_logistic.xlsx"

print(f"Base directory: {BASE_DIR}")
print(f"Result directory: {RESULT_PROJECT_DIR}")
print(f"Split data path: {SPLIT_DATA_PATH}")
for path_name in ["LOGISTIC_MODEL_PATH", "LOGISTIC_CONTRIB_PATH", "SAMPLE_PATH"]:
    print(f"{path_name}: {globals()[path_name]}")

## 3. Load Logistic Model Data

Load the Step 2 train/test split and the saved Logistic pipeline. Sampling uses Logistic probabilities.


In [ ]:
FEATURE_TO_NAME_MAP = {
    "home_ownership": "Home Ownership Status",
    "verification_status": "Income Verification Status",
    "purpose": "Loan Purpose",
    "area": "Borrower Area",
    "addr_state": "Borrower State",
    "term": "Number of Payments",
    "grade": "Loan Grade",
    "sub_grade": "Loan Sub Grade",
    "emp_length": "Employment Length",
    "pub_rec_bankruptcies": "Number of Public Bankruptcies",
    "funded_amnt": "Loan Amount",
    "int_rate": "Interest Rate",
    "installment": "Monthly Payment",
    "annual_inc": "Annual Income",
    "dti": "Debt to Income Ratio",
    "delinq_2yrs": "Number of Delinquencies in Past 2 Years",
    "fico_range_low": "Lowest FICO Score",
    "fico_range_high": "Highest FICO Score",
    "inq_last_6mths": "Number of Inquiries in Last 6 Months",
    "open_acc": "Number of Open Accounts",
    "pub_rec": "Number of Derogatory Public Records",
    "revol_bal": "Revolving Balance",
    "revol_util": "Revolving Utilization Rate",
    "total_acc": "Total Number of Accounts",
}
NAME_TO_FEATURE_MAP = {v: k for k, v in FEATURE_TO_NAME_MAP.items()}

In [ ]:
# if not SPLIT_DATA_PATH.exists():
#     raise FileNotFoundError(f"Missing split file: {SPLIT_DATA_PATH}. Run Step 2 first.")
# if not LOGISTIC_MODEL_PATH.exists():
#     raise FileNotFoundError(f"Missing Logistic model: {LOGISTIC_MODEL_PATH}. Run Step 2 first.")

with open(SPLIT_DATA_PATH, "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

logistic_pipe = joblib.load(LOGISTIC_MODEL_PATH)
logistic_train_prob = logistic_pipe.predict_proba(X_train)[:, 1]
logistic_test_prob = logistic_pipe.predict_proba(X_test)[:, 1]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")


## 4. Select The 500 Evaluation Samples

Find the F1-maximizing Logistic threshold, then sample cases across the confusion-matrix groups.


In [ ]:
def find_model_threshold(y_true, y_prob):
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)

    best_idx = int(np.nanargmax(f1_scores[:-1]))
    best_threshold = float(thresholds[best_idx])

    print(f"Best F1 threshold: {best_threshold:.3f}")
    print(f"Best F1 score: {f1_scores[best_idx]:.3f}")
    print(f"Precision: {precision[best_idx]:.3f}")
    print(f"Recall: {recall[best_idx]:.3f}")
    return best_threshold


def sample_loan_data(df, prob_col="PredProb", true_label_col="Label", threshold=0.5, n_samples=200, random_state=42):
    df = df.copy()
    df["PredictedClass"] = (df[prob_col] >= threshold).astype(int)
    df["Confidence"] = (df[prob_col] - threshold).abs()
    df["ConfidenceLevel"] = pd.cut(
        df["Confidence"],
        bins=[-np.inf, 0.15, 0.35, np.inf],
        labels=["low", "medium", "high"],
    )
    df["Stratum"] = (
        df[true_label_col].astype(str)
        + "_"
        + df["PredictedClass"].astype(str)
        + "_"
        + df["ConfidenceLevel"].astype(str)
    )

    selected_parts = []
    confusion_groups = [
        (1, 1, "true_positive"),
        (0, 0, "true_negative"),
        (0, 1, "false_positive"),
        (1, 0, "false_negative"),
    ]
    target_per_group = max(1, n_samples // len(confusion_groups))

    for true_label, pred_label, group_name in confusion_groups:
        group = df[(df[true_label_col] == true_label) & (df["PredictedClass"] == pred_label)]
        if len(group) == 0:
            print(f"No samples available for {group_name}.")
            continue
        selected_parts.append(group.sample(n=min(target_per_group, len(group)), random_state=random_state))

    selected = pd.concat(selected_parts).drop_duplicates() if selected_parts else pd.DataFrame(columns=df.columns)

    remaining_needed = n_samples - len(selected)
    if remaining_needed > 0:
        remaining = df.drop(index=selected.index, errors="ignore")
        if len(remaining) > 0:
            stratum_counts = remaining["Stratum"].value_counts(normalize=True)
            extra_parts = []
            for stratum, share in stratum_counts.items():
                group = remaining[remaining["Stratum"] == stratum]
                take = min(len(group), max(1, int(round(remaining_needed * share))))
                extra_parts.append(group.sample(n=take, random_state=random_state))
            if extra_parts:
                selected = pd.concat([selected] + extra_parts).drop_duplicates()

    if len(selected) < n_samples:
        remaining = df.drop(index=selected.index, errors="ignore")
        extra_n = min(n_samples - len(selected), len(remaining))
        if extra_n > 0:
            selected = pd.concat([selected, remaining.sample(n=extra_n, random_state=random_state)])

    if len(selected) > n_samples:
        selected = selected.sample(n=n_samples, random_state=random_state)

    selected = selected.drop(columns=["PredictedClass", "Confidence", "ConfidenceLevel", "Stratum"], errors="ignore")
    return selected.sort_values("Rowindex").reset_index(drop=True)


In [ ]:
best_threshold = find_model_threshold(y_train.astype(int), logistic_train_prob)
THRESHOLD = best_threshold if np.isfinite(best_threshold) else DEFAULT_THRESHOLD

# sample_pool = pd.DataFrame({
#     "Rowindex": X_test.index,
#     "PredProb": logistic_test_prob,
#     "Label": y_test.astype(int),
# })

# if RECREATE_SAMPLES or not SAMPLE_PATH.exists():
#     sample_df = sample_loan_data(
#         sample_pool,
#         prob_col="PredProb",
#         true_label_col="Label",
#         threshold=THRESHOLD,
#         n_samples=N_SAMPLES,
#         random_state=RANDOM_STATE,
#     )
#     sample_df.to_excel(SAMPLE_PATH, index=False)
# else:
sample_df = pd.read_excel(SAMPLE_PATH)

print(f"Selected samples: {len(sample_df)}")
print(f"Saved sample file: {SAMPLE_PATH}")
display(sample_df.head())


## 5. Load Logistic Contribution Reference

Load the wide Logistic contribution table from Step 3. This table is hidden from the LLM and used only for evaluation.


In [ ]:
if not LOGISTIC_CONTRIB_PATH.exists():
    raise FileNotFoundError(f"Missing Logistic contribution table: {LOGISTIC_CONTRIB_PATH}. Run Step 3 first.")

log_contrib = pd.read_excel(LOGISTIC_CONTRIB_PATH, index_col=0)
log_contrib.index = log_contrib.index.astype(int)

missing_rows = sorted(set(sample_df["Rowindex"].astype(int)) - set(log_contrib.index))
if missing_rows:
    raise KeyError(f"Sample rows missing from Logistic contribution table: {missing_rows[:10]}")

log_sample = log_contrib.loc[sample_df["Rowindex"].astype(int)].copy()
print(f"Logistic sample contribution shape: {log_sample.shape}")
display(log_sample.head())


## 6. Feature Names And Prompt Inputs

RQ2 gives the LLM borrower input values using readable feature names. The metric step maps returned readable names back to original feature names before comparing against the Logistic contribution ranking.


In [ ]:
FEATURE_TO_NAME_MAP = {
    "home_ownership": "Home Ownership Status",
    "verification_status": "Income Verification Status",
    "purpose": "Loan Purpose",
    "area": "Borrower Area",
    "addr_state": "Borrower State",
    "term": "Number of Payments",
    "grade": "Loan Grade",
    "sub_grade": "Loan Sub Grade",
    "emp_length": "Employment Length",
    "pub_rec_bankruptcies": "Number of Public Bankruptcies",
    "funded_amnt": "Loan Amount",
    "int_rate": "Interest Rate",
    "installment": "Monthly Payment",
    "annual_inc": "Annual Income",
    "dti": "Debt to Income Ratio",
    "delinq_2yrs": "Number of Delinquencies in Past 2 Years",
    "fico_range_low": "Lowest FICO Score",
    "fico_range_high": "Highest FICO Score",
    "inq_last_6mths": "Number of Inquiries in Last 6 Months",
    "open_acc": "Number of Open Accounts",
    "pub_rec": "Number of Derogatory Public Records",
    "revol_bal": "Revolving Balance",
    "revol_util": "Revolving Utilization Rate",
    "total_acc": "Total Number of Accounts",
}

FEATURE_ALIASES = {
    **{feature: feature for feature in FEATURE_TO_NAME_MAP},
    **{name: feature for feature, name in FEATURE_TO_NAME_MAP.items()},
    **{name.replace(" ", "_"): feature for feature, name in FEATURE_TO_NAME_MAP.items()},
    "Income_Verficiation_Status": "verification_status",
    "Income Verification": "verification_status",
    "Lend_Purpose": "purpose",
    "Loan Amount Requested": "funded_amnt",
    "Funded Amount": "funded_amnt",
}


def normalize_feature_text(text):
    text = str(text).strip()
    text = re.sub(r"[_\-]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.lower()


NORMALIZED_FEATURE_ALIASES = {
    normalize_feature_text(alias): feature
    for alias, feature in FEATURE_ALIASES.items()
}


def readable_feature_name(feature):
    return FEATURE_TO_NAME_MAP.get(feature, feature)


def normalize_feature_name(name):
    normalized = normalize_feature_text(name)
    return NORMALIZED_FEATURE_ALIASES.get(normalized)


In [ ]:
def format_feature_value(feature, value):
    if pd.isna(value):
        return "Missing"
    if feature in ["int_rate", "revol_util"]:
        return f"{float(value) * 100:.1f}%"
    if feature == "annual_inc":
        return f"{float(value):,.0f}"
    if isinstance(value, (np.floating, float)):
        return f"{float(value):.4g}"
    return str(value)


def prepare_input(rowindex, X, sample_df, threshold):
    rowindex = int(rowindex)
    feature_row = X.loc[rowindex]

    ## pred 
    pred_prob = sample_df[sample_df["Rowindex"]==rowindex]["PredProb"].values[0]


    lines = [
        "Feature                                  | Input Value",
        "-" * 72,
    ]
    for feature, value in feature_row.items():
        readable = readable_feature_name(feature)
        lines.append(f"{readable:<40s} | {format_feature_value(feature, value)}")

    return {
        "sample_index": rowindex,
        "predicted_probability": pred_prob,
        "input_table": "\n".join(lines),
        "allowed_readable_features": [readable_feature_name(feature) for feature in X.columns],
    }


example_sample = prepare_input(sample_df.loc[0, "Rowindex"], X_test, sample_df, THRESHOLD)
print(example_sample["allowed_readable_features"])


In [ ]:
print(example_sample["input_table"])

## 7. LiteLLM Setup

Set API keys from Colab userdata when available, otherwise use existing environment variables. Configure the model list here.


In [ ]:
# Run this manually in Colab if LiteLLM is not installed:
# !pip install litellm pandas tqdm google-generativeai -q

try:
    from google.colab import userdata
except Exception:
    userdata = None


# def set_env_from_colab(secret_name, env_name):
#     if os.environ.get(env_name):
#         return
#     if userdata is None:
#         return
#     try:
#         value = userdata.get(secret_name)
#         if value:
#             os.environ[env_name] = value
#     except Exception:
#         pass


# set_env_from_colab("OPENAI_API_KEY", "OPENAI_API_KEY")
# set_env_from_colab("Genimi_API", "GEMINI_API_KEY")
# set_env_from_colab("GEMINI_API_KEY", "GEMINI_API_KEY")
# set_env_from_colab("Calude_API", "ANTHROPIC_API_KEY")
# set_env_from_colab("ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY")

MODELS = [
    "gpt-4-turbo",
    "anthropic/claude-sonnet-4-5-20250929",
    "gemini/gemini-2.5-flash",
]

from dotenv import load_dotenv
import os

load_dotenv()

print("OpenAI:", bool(os.environ.get("OPENAI_API_KEY")))
print("Anthropic:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("Gemini:", bool(os.environ.get("GEMINI_API_KEY")))



## 8. Prompting And Response Parsing

All models receive the same input-only RQ2 task and are parsed into the same normalized table.


In [ ]:
OUTPUT_SCHEMA = {
    "ranked_features": [
        "Readable Feature Name 1",
        "Readable Feature Name 2",
        "Readable Feature Name 3",
    ]
}

def build_rq2_prompt(sample):
    allowed_features = "\n".join(f"- {feature}" for feature in sample["allowed_readable_features"])

    system_message = f"""
You are an expert credit risk analyst.

Your task is to infer which borrower input features most likely contributed to the Logistic Regression model prediction.
Rules:
1. Use ONLY the borrower input values provided below.
2. Do not use attribution values, SHAP values, or a model-provided ranking; none are shown.
3. Return the top {TOP_N_FEATURES} features from most important to least important.
4. Copy feature names exactly as listed in the provided feature list.
5. Do not invent, rename, merge, or explain features.
6. Return ONLY valid JSON, with no Markdown fences, prose, comments, or explanation.
7. Use exactly this schema:
{json.dumps(OUTPUT_SCHEMA, indent=2)}
""".strip()

    user_message = f"""
PREDICTED DEFAULT PROBABILITY: {sample['predicted_probability']:.4f}

BORROWER INPUT VALUES:
{sample['input_table']}

Task:
Rank the top {TOP_N_FEATURES} features that most likely contributed to this Logistic Regression model prediction.
""".strip()

    return system_message, user_message


def model_output_dir(model_name):
    return RQ2_LOGISTIC_DIR / f"Top_{TOP_N_FEATURES}" / model_name.replace("/", "__")

def api_key_for_model(model_name):
    if model_name.startswith("gemini/"):
        return os.environ.get("GEMINI_API_KEY")
    if model_name.startswith("claude") or model_name.startswith("anthropic/"):
        return os.environ.get("ANTHROPIC_API_KEY")
    return os.environ.get("OPENAI_API_KEY")


In [ ]:
def extract_json_block(text):
    text = str(text).strip()
    fence = chr(96) * 3
    text = re.sub(fence + r"(?:json)?", "", text, flags=re.IGNORECASE).replace(fence, "").strip()
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in LLM response.")
    return match.group(0)


def clean_json_text(text):
    text = text.strip()
    text = re.sub(r",\s*([}\]])", r"", text)
    text = text.replace("NaN", "null").replace("Infinity", "null").replace("-Infinity", "null")
    return text


def repair_missing_commas(text):
    text = re.sub(r"}\s*{", r"},{", text)
    text = re.sub(
        r'(?P<val>(?:true|false|null|-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?|"[^"\]*(?:\.[^"\]*)*"|\]|\}))\s+(?P<key>"[^"]+"\s*:)',
        r'\g<val>, \g<key>',
        text,
    )
    return text


def safe_json_loads(raw_text):
    try:
        return json.loads(raw_text)
    except Exception:
        pass

    extracted = extract_json_block(raw_text)
    cleaned = clean_json_text(extracted)
    try:
        return json.loads(cleaned)
    except Exception:
        repaired = clean_json_text(repair_missing_commas(cleaned))
        return json.loads(repaired)


def extract_ranked_features_from_text(raw_text):
    """Fallback for models that return a plain ranked list instead of JSON."""
    found = []
    seen = set()

    for line in str(raw_text).splitlines():
        cleaned = re.sub(r"^\s*(?:[-*]|\d+[.)]|rank\s*\d+[:.)-]?)\s*", "", line, flags=re.IGNORECASE)
        cleaned = cleaned.strip().strip("\"'`,;")
        if not cleaned:
            continue

        matched = normalize_feature_name(cleaned)
        if matched is None:
            normalized_line = normalize_feature_text(cleaned)
            for alias_key, feature in NORMALIZED_FEATURE_ALIASES.items():
                if alias_key and alias_key in normalized_line:
                    matched = feature
                    break

        if matched is not None and matched not in seen:
            found.append(matched)
            seen.add(matched)

    return found


def normalize_ranked_features(parsed):
    ranked = parsed.get("ranked_features")
    if ranked is None:
        raise KeyError("Missing 'ranked_features' in response JSON.")

    features = []
    for item in ranked:
        if isinstance(item, str):
            features.append(item)
        elif isinstance(item, dict):
            feature = item.get("Feature") or item.get("feature") or item.get("name")
            if feature is not None:
                features.append(feature)

    if not features:
        raise ValueError("No ranked features could be parsed from response.")
    return features


In [ ]:
def call_llm_with_retry(model_name, system_message, user_message, max_retries=6, timeout=90, min_sleep=0.8, max_tokens=800):
    from litellm import completion

    last_error = None
    api_key = api_key_for_model(model_name)

    for attempt in range(max_retries):
        if min_sleep:
            time.sleep(min_sleep)
        try:
            kwargs = {
                "model": model_name,
                "messages": [
                    {"role": "system", "content": system_message},
                    {"role": "user", "content": user_message},
                ],
                "temperature": 0,
                "max_tokens": max_tokens,
                "timeout": timeout,
            }
            if api_key:
                kwargs["api_key"] = api_key
            if not model_name.startswith("claude") and not model_name.startswith("anthropic/"):
                kwargs["response_format"] = {"type": "json_object"}

            response = completion(**kwargs)
            content = response.choices[0].message.content
            if content is None or str(content).strip() == "":
                raise ValueError("Empty LLM response.")
            return {"success": True, "response": content, "error": None}
        except Exception as error:
            last_error = error
            backoff = min(30, (2 ** attempt) * 0.8) + random.random() * 0.3
            time.sleep(backoff)

    return {"success": False, "response": None, "error": str(last_error)}


def parse_llm_response(raw_response, sample_index):
    try:
        parsed = safe_json_loads(raw_response)
        ranked_features = normalize_ranked_features(parsed)
    except Exception as json_error:
        normalized = extract_ranked_features_from_text(raw_response)
        if not normalized:
            raise json_error
    else:
        normalized = []
        unknown = []
        for feature in ranked_features:
            original = normalize_feature_name(feature)
            if original is None:
                unknown.append(feature)
            elif original not in normalized:
                normalized.append(original)

        if unknown:
            raise ValueError(f"Response includes unknown features: {unknown}")

    if not normalized:
        raise ValueError("No valid feature names after normalization.")

    return pd.DataFrame({
        "Rowindex": sample_index,
        "FeatureRank": range(1, len(normalized) + 1),
        "Feature": normalized,
        "ReadableFeature": [readable_feature_name(feature) for feature in normalized],
    })


## 9. Run LLM Ranking Experiment

Run this cell once per model or for all models. Results are saved in batches so partial progress is preserved.


In [ ]:
def run_rq2_model(model_name, sample_df, X, threshold, batch_size=BATCH_SIZE):
    save_dir = model_output_dir(model_name)
    save_dir.mkdir(parents=True, exist_ok=True)
    error_log_path = save_dir / "errors.jsonl"

    batch_rows = []
    batch_number = 0
    all_errors = []

    for position, rowindex in enumerate(sample_df["Rowindex"].astype(int), start=1):
        sample = prepare_input(rowindex, X, sample_df, threshold)
        system_message, user_message = build_rq2_prompt(sample)
        result = call_llm_with_retry(model_name, system_message, user_message)

        if not result["success"]:
            error = {"Rowindex": rowindex, "ErrorType": "API_ERROR", "Error": result["error"]}
            all_errors.append(error)
            print(f"API error at sample {rowindex}: {result['error']}")
            continue

        try:
            parsed_df = parse_llm_response(result["response"], rowindex)
            batch_rows.append(parsed_df)
        except Exception as error:
            error_record = {
                "Rowindex": rowindex,
                "ErrorType": "FORMAT_ERROR",
                "Error": str(error),
                "RawResponse": result["response"],
            }
            all_errors.append(error_record)
            print(f"Format error at sample {rowindex}: {error}")
            continue

        if len(batch_rows) >= batch_size:
            batch_number += 1
            batch_df = pd.concat(batch_rows, ignore_index=True)
            batch_path = save_dir / f"batch_{batch_number}.xlsx"
            batch_df.to_excel(batch_path, index=False)
            print(f"Saved {batch_path}")
            batch_rows = []

    if batch_rows:
        batch_number += 1
        batch_df = pd.concat(batch_rows, ignore_index=True)
        batch_path = save_dir / f"batch_{batch_number}.xlsx"
        batch_df.to_excel(batch_path, index=False)
        print(f"Saved {batch_path}")

    if all_errors:
        with open(error_log_path, "w", encoding="utf-8") as f:
            for error in all_errors:
                f.write(json.dumps(error, ensure_ascii=False) + "\n")
        print(f"Saved error log: {error_log_path}")

    return save_dir


# Example usage:
# for model_name in MODELS:
#     run_rq2_model(model_name, sample_df, X_test, THRESHOLD)

In [ ]:
sample = prepare_input(33, X_test, sample_df, THRESHOLD)
system_message, user_message = build_rq2_prompt(sample)
print(system_message)

### When the top K feature is 4

In [ ]:

model_name = MODELS[0]  # Change index to run different models
run_rq2_model(model_name, sample_df, X_test, THRESHOLD)

In [ ]:
model_name = MODELS[1]  # Change index to run different models
run_rq2_model(model_name, sample_df, X_test, THRESHOLD)

In [ ]:
model_name = MODELS[2]  # Change index to run different models
run_rq2_model(model_name, sample_df, X_test, THRESHOLD)

### When the top K feature is 24 

In [ ]:

model_name = MODELS[0]  # Change index to run different models
run_rq2_model(model_name, sample_df, X_test, THRESHOLD)

In [ ]:

model_name = MODELS[1]  # Change index to run different models
run_rq2_model(model_name, sample_df, X_test, THRESHOLD)

In [ ]:

model_name = MODELS[2]  # Change index to run different models
run_rq2_model(model_name, sample_df, X_test, THRESHOLD)

## 10. Metrics

Compare each LLM ranking against the hidden Logistic contribution reference ranking.


In [ ]:
def overlap_topk(llm_ranking, ref_ranking, k=TOP_N_FEATURES):
    llm_topk = set(llm_ranking[:k])
    ref_topk = set(ref_ranking[:k])
    if not llm_topk:
        return 0.0
    return len(llm_topk & ref_topk) / k


def kendall_tau_topk(llm_ranking, ref_ranking, k=TOP_N_FEATURES):
    llm_topk = llm_ranking[:k]
    ref_rank_map = {feature: rank for rank, feature in enumerate(ref_ranking, start=1)}
    common_features = [feature for feature in llm_topk if feature in ref_rank_map]

    if len(common_features) < 2:
        return np.nan

    llm_ranks = list(range(1, len(common_features) + 1))
    ref_ranks = [ref_rank_map[feature] for feature in common_features]
    tau, _ = kendalltau(llm_ranks, ref_ranks)
    return tau


def bootstrap_metric_ci(metric_df, model_name=None, n_bootstrap=2000, ci=0.95, random_state=42):
    """Bootstrap instance-level confidence intervals for mean ranking metrics."""
    if metric_df.empty:
        return pd.DataFrame(columns=[
            "Model", "Metric", "Mean", "CI_Lower", "CI_Upper",
            "N", "BootstrapSamples", "CI_Level"
        ])

    rng = np.random.default_rng(random_state)
    alpha = 1 - ci
    numeric_cols = metric_df.select_dtypes(include="number").columns.tolist()
    metric_cols = [col for col in numeric_cols if col != "Rowindex"]

    rows = []
    for metric in metric_cols:
        values = pd.to_numeric(metric_df[metric], errors="coerce").dropna().to_numpy(dtype=float)
        n = len(values)
        if n == 0:
            rows.append({
                "Model": model_name,
                "Metric": metric,
                "Mean": np.nan,
                "CI_Lower": np.nan,
                "CI_Upper": np.nan,
                "N": 0,
                "BootstrapSamples": n_bootstrap,
                "CI_Level": ci,
            })
            continue

        bootstrap_indices = rng.integers(0, n, size=(n_bootstrap, n))
        bootstrap_means = values[bootstrap_indices].mean(axis=1)
        lower, upper = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])

        rows.append({
            "Model": model_name,
            "Metric": metric,
            "Mean": float(values.mean()),
            "CI_Lower": float(lower),
            "CI_Upper": float(upper),
            "N": n,
            "BootstrapSamples": n_bootstrap,
            "CI_Level": ci,
        })

    return pd.DataFrame(rows)

def load_llm_batches(result_dir):
    files = sorted(Path(result_dir).glob("batch_*.xlsx"))
    if not files:
        raise FileNotFoundError(f"No batch files found in {result_dir}")
    frames = [pd.read_excel(file) for file in files]
    result = pd.concat(frames, ignore_index=True)
    result = result[["Rowindex", "FeatureRank", "Feature", "ReadableFeature"]].copy()
    result["Rowindex"] = result["Rowindex"].astype(int)
    result = result.sort_values(["Rowindex", "FeatureRank"]).reset_index(drop=True)
    return result


def reference_rank_for_row(rowindex, contribution_df):
    row = contribution_df.loc[int(rowindex)].copy()
    return row.abs().sort_values(ascending=False).index.tolist()


In [ ]:
def evaluate_rq2_results(model_name, contribution_df, sample_df, k_values=(3, 5, 10)):
    result_dir = model_output_dir(model_name)
    llm_results = load_llm_batches(result_dir)
    llm_results.to_excel(result_dir / "summary.xlsx", index=False)

    rows = []
    for rowindex in sample_df["Rowindex"].astype(int):
        model_rows = llm_results[llm_results["Rowindex"] == rowindex].sort_values("FeatureRank")
        if model_rows.empty:
            continue

        llm_rank = model_rows["Feature"].tolist()
        ref_rank = reference_rank_for_row(rowindex, contribution_df)

        row = {"Rowindex": rowindex, "Model": model_name}
        for k in k_values:
            row[f"overlap_top{k}"] = overlap_topk(llm_rank, ref_rank, k=k)
            row[f"kendall_top{k}"] = kendall_tau_topk(llm_rank, ref_rank, k=k)
        rows.append(row)

    metric_df = pd.DataFrame(rows)
    metric_path = result_dir / "metric.xlsx"
    metric_df.to_excel(metric_path, index=False)
    print(f"Saved metric file: {metric_path}")
    return metric_df


# Example usage after running LLM calls:
# metric_tables = {model_name: evaluate_rq2_results(model_name, log_contrib, sample_df) for model_name in MODELS}


In [ ]:
metric_tables = {model_name: evaluate_rq2_results(model_name, log_contrib, sample_df) for model_name in MODELS}

## Feature-Name-Only Control

This optional control removes borrower-specific values and asks the LLM to rank generally important features from the same allowed list. Compare this against the main RQ2 condition to assess whether performance is driven by sample-specific values or generic credit-risk priors.


In [ ]:
def feature_name_only_control_output_dir(model_name):
    return RQ2_LOGISTIC_DIR / "feature_name_only_control" / model_name.replace("/", "__")


def prepare_feature_name_only_control_input(rowindex, X):
    return {
        "sample_index": int(rowindex),
        "allowed_readable_features": [readable_feature_name(feature) for feature in X.columns],
    }


def build_rq2_feature_name_only_control_prompt(sample):
    allowed_features = "\n".join(f"- {feature}" for feature in sample["allowed_readable_features"])

    system_message = f"""
You are an expert credit risk analyst.

This is a control condition. You are given only the feature names, not borrower-specific input values.
Your task is to rank the top {TOP_N_FEATURES} features that you would generally expect to influence a Logistic Regression default prediction.
Rules:
1. Use ONLY the allowed feature list.
2. Return the top {TOP_N_FEATURES} features from most important to least important.
3. Do not invent, rename, merge, or explain features.
4. Return ONLY valid JSON, with no Markdown fences, prose, comments, or explanation.
5. Use exactly this schema:
{json.dumps(OUTPUT_SCHEMA, indent=2)}
""".strip()

    user_message = f"""
APPLICATION ID: {sample['sample_index']}

ALLOWED FEATURE NAMES:
{allowed_features}

Task:
Rank the top {TOP_N_FEATURES} generally important features for a Logistic Regression default prediction.
""".strip()

    return system_message, user_message


def run_rq2_feature_name_only_control_model(model_name, sample_df, X, batch_size=BATCH_SIZE):
    save_dir = feature_name_only_control_output_dir(model_name)
    save_dir.mkdir(parents=True, exist_ok=True)
    error_log_path = save_dir / "errors.jsonl"

    batch_rows = []
    batch_number = 0
    all_errors = []

    for position, rowindex in enumerate(sample_df["Rowindex"].astype(int), start=1):
        sample = prepare_feature_name_only_control_input(rowindex, X)
        system_message, user_message = build_rq2_feature_name_only_control_prompt(sample)
        result = call_llm_with_retry(model_name, system_message, user_message)

        if not result["success"]:
            error = {"Rowindex": rowindex, "ErrorType": "API_ERROR", "Error": result["error"]}
            all_errors.append(error)
            print(f"API error at sample {rowindex}: {result['error']}")
            continue

        try:
            parsed_df = parse_llm_response(result["response"], rowindex)
            batch_rows.append(parsed_df)
        except Exception as error:
            error_record = {
                "Rowindex": rowindex,
                "ErrorType": "FORMAT_ERROR",
                "Error": str(error),
                "RawResponse": result["response"],
            }
            all_errors.append(error_record)
            print(f"Format error at sample {rowindex}: {error}")
            continue

        if len(batch_rows) >= batch_size:
            batch_number += 1
            batch_df = pd.concat(batch_rows, ignore_index=True)
            batch_path = save_dir / f"batch_{batch_number}.xlsx"
            batch_df.to_excel(batch_path, index=False)
            print(f"Saved {batch_path}")
            batch_rows = []

    if batch_rows:
        batch_number += 1
        batch_df = pd.concat(batch_rows, ignore_index=True)
        batch_path = save_dir / f"batch_{batch_number}.xlsx"
        batch_df.to_excel(batch_path, index=False)
        print(f"Saved {batch_path}")

    if all_errors:
        with open(error_log_path, "w", encoding="utf-8") as f:
            for error in all_errors:
                f.write(json.dumps(error, ensure_ascii=False) + "\n")
        print(f"Saved error log: {error_log_path}")

    return save_dir


def evaluate_rq2_feature_name_only_control_results(model_name, attribution_df, sample_df, k_values=(3, 5, 10)):
    result_dir = feature_name_only_control_output_dir(model_name)
    llm_results = load_llm_batches(result_dir)
    llm_results.to_excel(result_dir / "summary.xlsx", index=False)

    rows = []
    for rowindex in sample_df["Rowindex"].astype(int):
        model_rows = llm_results[llm_results["Rowindex"] == rowindex].sort_values("FeatureRank")
        if model_rows.empty:
            continue

        llm_rank = model_rows["Feature"].tolist()
        ref_rank = reference_rank_for_row(rowindex, attribution_df)

        row = {"Rowindex": rowindex, "Model": model_name, "Condition": "feature_name_only_control"}
        for k in k_values:
            row[f"overlap_top{k}"] = overlap_topk(llm_rank, ref_rank, k=k)
            row[f"kendall_top{k}"] = kendall_tau_topk(llm_rank, ref_rank, k=k)
        rows.append(row)

    metric_df = pd.DataFrame(rows)
    metric_path = result_dir / "metric.xlsx"
    metric_df.to_excel(metric_path, index=False)
    print(f"Saved metric file: {metric_path}")
    return metric_df


# Example usage after main RQ2 runs:
# for model_name in MODELS:
#     run_rq2_feature_name_only_control_model(model_name, sample_df, X_test)
# control_metric_tables = {model_name: evaluate_rq2_feature_name_only_control_results(model_name, log_contrib, sample_df) for model_name in MODELS}


## 11. Summarize Metrics

Load metric files for all completed models and compare average performance.


In [ ]:
def summarize_completed_metrics(model_names):
    frames = []
    for model_name in model_names:
        metric_path = model_output_dir(model_name) / "metric.xlsx"
        if not metric_path.exists():
            print(f"Skipping missing metric file: {metric_path}")
            continue
        frame = pd.read_excel(metric_path)
        frame["Model"] = model_name
        frames.append(frame)

    if not frames:
        return pd.DataFrame()

    all_metrics = pd.concat(frames, ignore_index=True)
    summary = all_metrics.groupby("Model").mean(numeric_only=True).reset_index()
    display(summary)
    return summary


summary_df = summarize_completed_metrics(MODELS)


In [ ]:
def summarize_completed_metrics(model_names):
    frames = []
    for model_name in model_names:
        metric_path = model_output_dir(model_name) / "metric.xlsx"
        if not metric_path.exists():
            print(f"Skipping missing metric file: {metric_path}")
            continue
        frame = pd.read_excel(metric_path)
        frame["Model"] = model_name
        frames.append(frame)

    if not frames:
        return pd.DataFrame()

    all_metrics = pd.concat(frames, ignore_index=True)
    summary = all_metrics.groupby("Model").mean(numeric_only=True).reset_index()
    display(summary)
    return summary


summary_df = summarize_completed_metrics(MODELS)

## Bootstrap Confidence Intervals

Run this optional section after `metric.xlsx` files have been generated. It resamples instances/rows within each metric file to estimate confidence intervals for Overlap@K and Kendall tau metrics.

In [ ]:
def bootstrap_metric_ci(metric_df, model_name=None, n_bootstrap=2000, ci=0.95, random_state=42):
    """Bootstrap instance-level confidence intervals for mean ranking metrics."""
    if metric_df.empty:
        return pd.DataFrame(columns=[
            "Model", "Metric", "Mean", "CI_Lower", "CI_Upper",
            "N", "BootstrapSamples", "CI_Level"
        ])

    rng = np.random.default_rng(random_state)
    alpha = 1 - ci
    numeric_cols = metric_df.select_dtypes(include="number").columns.tolist()
    metric_cols = [col for col in numeric_cols if col != "Rowindex"]

    rows = []
    for metric in metric_cols:
        values = pd.to_numeric(metric_df[metric], errors="coerce").dropna().to_numpy(dtype=float)
        n = len(values)
        if n == 0:
            rows.append({
                "Model": model_name,
                "Metric": metric,
                "Mean": np.nan,
                "CI_Lower": np.nan,
                "CI_Upper": np.nan,
                "N": 0,
                "BootstrapSamples": n_bootstrap,
                "CI_Level": ci,
            })
            continue

        bootstrap_indices = rng.integers(0, n, size=(n_bootstrap, n))
        bootstrap_means = values[bootstrap_indices].mean(axis=1)
        lower, upper = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])

        rows.append({
            "Model": model_name,
            "Metric": metric,
            "Mean": float(values.mean()),
            "CI_Lower": float(lower),
            "CI_Upper": float(upper),
            "N": n,
            "BootstrapSamples": n_bootstrap,
            "CI_Level": ci,
        })

    return pd.DataFrame(rows)


def generate_bootstrap_ci_for_completed_metrics(model_names, n_bootstrap=2000, ci=0.95, random_state=42):
    ci_frames = []
    for model_name in model_names:
        result_dir = model_output_dir(model_name)
        metric_path = result_dir / "metric.xlsx"
        if not metric_path.exists():
            print(f"Skipping missing metric file: {metric_path}")
            continue

        metric_df = pd.read_excel(metric_path)
        ci_df = bootstrap_metric_ci(
            metric_df,
            model_name=model_name,
            n_bootstrap=n_bootstrap,
            ci=ci,
            random_state=random_state,
        )
        ci_path = result_dir / "metric_bootstrap_ci.xlsx"
        ci_df.to_excel(ci_path, index=False)
        print(f"Saved bootstrap CI file: {ci_path}")
        ci_frames.append(ci_df)

    if not ci_frames:
        return pd.DataFrame()

    ci_summary = pd.concat(ci_frames, ignore_index=True)
    display(ci_summary)
    return ci_summary


# Example usage after metric.xlsx files exist:
# ci_summary_df = generate_bootstrap_ci_for_completed_metrics(MODELS)